# Spatial2D Inference

This notebook uses `run/.../last.ckpt` as the pretrained encoder, keeps the simulator initial condition from the sample `txt`, keeps the observed final state from the raw image when available, and reproduces the same style of diagnostics as the earlier inference tutorials.


In [ ]:
import re
from pathlib import Path

import rootutils

rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from omegaconf import OmegaConf
from PIL import Image

from src.viaABC.systems import Spatial2D

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


In [ ]:
PROJECT_ROOT = Path(rootutils.find_root(Path.cwd(), indicator=".project-root"))
SAMPLE_ID = "sample_1"
TOY_GRID_SIZE = 10

BASE_INFERENCE_CFG = OmegaConf.load(PROJECT_ROOT / "configs" / "inference.yaml")
SPATIAL2D_INFERENCE_CFG = OmegaConf.load(PROJECT_ROOT / "configs" / "inference" / "spatial2D.yaml")
DATA_CFG = OmegaConf.load(PROJECT_ROOT / "configs" / "data" / "spatial2D.yaml")
INFERENCE_CFG = OmegaConf.merge(BASE_INFERENCE_CFG, SPATIAL2D_INFERENCE_CFG)
INFERENCE_CFG.data = DATA_CFG
INFERENCE_CFG.paths = {"root_dir": str(PROJECT_ROOT), "data_dir": str(PROJECT_ROOT / "data")}

TOY_ABC_CFG = OmegaConf.create({
    "num_particles": 4,
    "k": 2,
    "q_threshold": 0.90,
    "max_generations": 1,
})

PARAM_NAMES = ["alpha", "beta", "gamma"]
STATE_CMAP = ListedColormap([
    "#d73027",  # 0
    "#fee08b",  # 1
    "#4575b4",  # 2
    "#2b1744",  # 3 background
    "#1a9850",  # 4
    "#d9d9d9",  # 5 fallback
])

print("Loaded Spatial2D inference config:")
print(OmegaConf.to_yaml(INFERENCE_CFG.abc))
print("Toy ABC override:")
print(OmegaConf.to_yaml(TOY_ABC_CFG))


class DummyModel:
    device = torch.device("cpu")

    def eval(self):
        return self


class ToySpatial2D(Spatial2D):
    def get_latent(self, x):
        x = np.asarray(x.detach().cpu() if isinstance(x, torch.Tensor) else x)

        if x.ndim == 2:
            x = self.labels2map(x)[None, ...]
        elif x.ndim == 3:
            if x.shape[0] == 6:
                x = x[None, ...]
            else:
                x = np.stack([self.labels2map(grid) for grid in x], axis=0)
        elif x.ndim != 4:
            raise ValueError(f"Unsupported toy latent input shape: {x.shape}")

        batch, channels, height, width = x.shape
        crop_h = height - (height % TOY_GRID_SIZE)
        crop_w = width - (width % TOY_GRID_SIZE)
        x = x[:, :, :crop_h, :crop_w]
        block_h = crop_h // TOY_GRID_SIZE
        block_w = crop_w // TOY_GRID_SIZE
        x = x.reshape(batch, channels, TOY_GRID_SIZE, block_h, TOY_GRID_SIZE, block_w)
        x = x.mean(axis=(3, 5)).transpose(0, 2, 3, 1)
        return x.reshape(batch, TOY_GRID_SIZE * TOY_GRID_SIZE, channels).astype(np.float32)


def load_sample_cfg(sample_id=SAMPLE_ID, cfg=INFERENCE_CFG):
    data_dir = PROJECT_ROOT / "data"
    sample_cfg = OmegaConf.to_container(cfg.data.observation_samples[sample_id], resolve=False)
    resolved = {}
    for key, value in sample_cfg.items():
        if isinstance(value, str):
            value = value.replace("${paths.data_dir}", str(data_dir))
        resolved[key] = value
    return resolved


def resolve_sample_paths(sample_id=SAMPLE_ID):
    sample_cfg = load_sample_cfg(sample_id)
    txt_path = Path(sample_cfg["txt"])
    if not txt_path.is_absolute():
        txt_path = PROJECT_ROOT / txt_path

    image_candidates = []
    image_value = sample_cfg.get("image")
    if image_value is not None:
        image_path = Path(image_value)
        image_candidates.append(image_path if image_path.is_absolute() else PROJECT_ROOT / image_path)

    match = re.search(r"(\d+)$", sample_id)
    if match is not None:
        idx = match.group(1)
        image_candidates.extend([
            PROJECT_ROOT / "data" / "spatial2D" / f"img{idx}.jpg",
            PROJECT_ROOT / "data" / "spatial2D" / f"image{idx}_processed.jpg",
            PROJECT_ROOT / "data" / "spatial2D" / f"image{idx}.jpg",
        ])

    raw_image_path = next((path for path in image_candidates if path.exists()), None)
    return txt_path, raw_image_path


def build_spatial2d_system(sample_id=SAMPLE_ID, cfg=INFERENCE_CFG):
    return ToySpatial2D(
        model=DummyModel(),
        sample_id=sample_id,
        pooling_method=cfg.system.pooling_method,
        metric=cfg.system.metric,
    )


def weighted_mean_std(particles, weights):
    mean = np.average(particles, axis=0, weights=weights)
    var = np.average((particles - mean) ** 2, axis=0, weights=weights)
    return mean, np.sqrt(var)


def weighted_quantile(values, quantiles, weights):
    values = np.asarray(values)
    quantiles = np.asarray(quantiles)
    weights = np.asarray(weights)
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cumulative = np.cumsum(weights) - 0.5 * weights
    cumulative = cumulative / weights.sum()
    return np.interp(quantiles, cumulative, values)


def draw_grid(ax, grid, title):
    ax.imshow(grid, cmap=STATE_CMAP, vmin=0, vmax=5)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])


def plot_weighted_corner(particles, weights, param_names):
    dim = particles.shape[1]
    fig, axes = plt.subplots(dim, dim, figsize=(4 * dim, 4 * dim))
    for row in range(dim):
        for col in range(dim):
            ax = axes[row, col]
            if row < col:
                ax.axis("off")
                continue
            if row == col:
                ax.hist(particles[:, col], bins=12, weights=weights, color="#6d6d6d", edgecolor="white")
                mean = np.average(particles[:, col], weights=weights)
                ax.axvline(mean, color="#b22222", linestyle="--", linewidth=1.5)
                ax.set_ylabel("Weighted count")
            else:
                ax.hist2d(particles[:, col], particles[:, row], bins=12, weights=weights, cmap="viridis")
            if row == dim - 1:
                ax.set_xlabel(param_names[col])
            if col == 0 and row > 0:
                ax.set_ylabel(param_names[row])
    fig.suptitle("Toy posterior joint and marginal distributions", y=1.02)
    fig.tight_layout()
    return fig


def plot_generation_estimates(generations, param_names):
    means = []
    medians = []
    for generation in generations:
        particles = generation["particles"]
        weights = generation["weights"]
        means.append(np.average(particles, axis=0, weights=weights))
        medians.append([
            weighted_quantile(particles[:, idx], 0.5, weights)
            for idx in range(particles.shape[1])
        ])
    means = np.asarray(means)
    medians = np.asarray(medians)
    xs = np.arange(len(generations))

    fig, axes = plt.subplots(len(param_names), 1, figsize=(9, 2.7 * len(param_names)), sharex=True)
    if len(param_names) == 1:
        axes = [axes]
    for idx, ax in enumerate(axes):
        ax.plot(xs, means[:, idx], marker="o", label="weighted mean")
        ax.plot(xs, medians[:, idx], marker="s", label="weighted median")
        ax.set_ylabel(param_names[idx])
        ax.grid(alpha=0.2)
    axes[-1].set_xlabel("Generation")
    axes[0].legend(loc="best")
    fig.suptitle("Toy posterior estimates by generation", y=1.02)
    fig.tight_layout()
    return fig


In [ ]:
txt_path, raw_image_path = resolve_sample_paths(SAMPLE_ID)
system = build_spatial2d_system(sample_id=SAMPLE_ID)

initial_grid = np.loadtxt(txt_path, dtype=np.uint8)
observed_grid = system._observation_grids[0]
raw_image = np.array(Image.open(raw_image_path)) if raw_image_path is not None else None
raw_image_grid = system.image_to_grid(raw_image) if raw_image is not None else None

latent_preview = system.encoded_observational_data[0].reshape(TOY_GRID_SIZE, TOY_GRID_SIZE, -1).argmax(axis=-1)

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
draw_grid(axes[0, 0], initial_grid, "Initial state from txt")
if raw_image is not None:
    draw_grid(axes[0, 1], raw_image_grid, "Observed final state (raw image -> grid)")
else:
    draw_grid(axes[0, 1], observed_grid, "Observed final state")
draw_grid(axes[1, 0], observed_grid, "Observed final state used by toy encoder")
draw_grid(axes[1, 1], latent_preview, f"Toy latent preview ({TOY_GRID_SIZE}x{TOY_GRID_SIZE})")
fig.tight_layout()
plt.show()

print("txt path:", txt_path)
print("raw image path:", raw_image_path)
print("toy encoded observation shape:", system.encoded_observational_data.shape)


In [ ]:
abc_system = build_spatial2d_system(sample_id=SAMPLE_ID)
abc_system.run(
    num_particles=TOY_ABC_CFG.num_particles,
    k=TOY_ABC_CFG.k,
    q_threshold=TOY_ABC_CFG.q_threshold,
    max_generations=TOY_ABC_CFG.max_generations,
)

last_generation = abc_system.generations[-1]
posterior_particles = last_generation["particles"]
posterior_weights = last_generation["weights"]

posterior_mean, posterior_std = weighted_mean_std(posterior_particles, posterior_weights)
posterior_median = np.array([
    weighted_quantile(posterior_particles[:, idx], 0.5, posterior_weights)
    for idx in range(posterior_particles.shape[1])
])
map_params = posterior_particles[np.argmax(posterior_weights)]

print("Finished generations:", len(abc_system.generations))
print("Last epsilon:", float(last_generation["epsilon"]))
print("Posterior mean :", np.round(posterior_mean, 4))
print("Posterior std  :", np.round(posterior_std, 4))
print("Posterior median:", np.round(posterior_median, 4))
print("MAP particle   :", np.round(map_params, 4))


In [ ]:
posterior_fig = plot_weighted_corner(posterior_particles, posterior_weights, PARAM_NAMES)
plt.show()


In [ ]:
posterior_mean_sim, _ = abc_system.simulate(posterior_mean)
map_sim, _ = abc_system.simulate(map_params)

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
draw_grid(axes[0, 0], initial_grid, "Initial state from txt")
if raw_image is not None:
    draw_grid(axes[0, 1], raw_image_grid, "Observed final state (raw image -> grid)")
else:
    draw_grid(axes[0, 1], observed_grid, "Observed final state")
draw_grid(axes[0, 2], observed_grid, "Observed final state used by encoder")
draw_grid(axes[1, 0], recon_grid, "Encoder reconstruction")
draw_grid(axes[1, 1], posterior_mean_sim, "Inferred final state (posterior mean)")
draw_grid(axes[1, 2], map_sim, "Inferred final state (MAP particle)")
fig.suptitle(
    "Observed vs inferred Spatial2D states\n"
    + f"posterior mean = {np.round(posterior_mean, 4)}, MAP = {np.round(map_params, 4)}",
    y=1.02,
)
fig.tight_layout()
plt.show()


In [ ]:
estimate_fig = plot_generation_estimates(abc_system.generations, PARAM_NAMES)
plt.show()
